# Motion-S: Text-to-Sign Motion Generation
## Hybrid Pipeline: Semantic Retrieval + Progressive Transformer + Diffusion Refiner

| Tier | Method | When used |
|------|--------|----------|
| 1 | TF-IDF + SBERT dual-encoder kNN | Retrieval score ≥ 0.75 |
| 2 | Progressive Transformer (seq2seq) | Otherwise |
| 2.5 | Blended (retrieval + generation) | 0.40 ≤ score < 0.75 |
| 3 | Diffusion refiner (optional, GPU) | Post-generation |
| ✓ | Cosine smoothing + root normalisation | Always |

In [ ]:
# Cell 1 — Install dependencies
!pip install -q sentence-transformers dtw-python sacrebleu einops
!pip install -q transformers accelerate

In [ ]:
# Cell 2 — Run the full pipeline
# (paste the entire motion_s_pipeline.py content here, then call main())
exec(open('/kaggle/input/motion-s-pipeline/motion_s_pipeline.py').read())
main()

In [ ]:
# Cell 3 — Verify submission
import os, numpy as np
from pathlib import Path

sub_dir = Path('/kaggle/working/submission')
files   = list(sub_dir.glob('*.npy'))
print(f'Generated {len(files)} pose files')

# Check a sample
sample = np.load(str(files[0]))
print(f'Sample shape: {sample.shape}  (T, 128, 3)')
print(f'Value range: min={sample.min():.3f}  max={sample.max():.3f}')

In [ ]:
# Cell 4 — Visualise a generated pose sequence (optional)
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from IPython.display import HTML

# DWPose body skeleton connections (18 body joints)
SKELETON = [
    (0,1),(1,2),(2,3),(3,4),          # right arm
    (0,5),(5,6),(6,7),(7,8),          # left arm
    (0,9),(9,10),(10,11),(11,12),     # torso + right leg (approx)
    (0,13),(13,14),(14,15),(15,16),   # left leg (approx)
]

def animate_pose(pose_npy_path: str, fps: int = 15):
    pose = np.load(pose_npy_path)   # (T, 128, 3)
    T    = pose.shape[0]
    fig, ax = plt.subplots(figsize=(5, 6))
    ax.set_xlim(-2, 2); ax.set_ylim(-2, 2)
    ax.set_aspect('equal'); ax.axis('off')
    ax.invert_yaxis()

    scat  = ax.scatter([], [], c='royalblue', s=15, zorder=3)
    lines = [ax.plot([], [], 'k-', lw=1.5)[0] for _ in SKELETON]

    def update(t):
        xy = pose[t, :18, :2]       # body joints only for viz
        scat.set_offsets(xy)
        for (a, b), ln in zip(SKELETON, lines):
            ln.set_data([xy[a,0], xy[b,0]], [xy[a,1], xy[b,1]])
        ax.set_title(f'Frame {t+1}/{T}')
        return [scat] + lines

    ani = animation.FuncAnimation(fig, update, frames=T,
                                   interval=1000//fps, blit=True)
    plt.close()
    return HTML(ani.to_jshtml())

# Show first generated test sample
sample_path = str(sorted(sub_dir.glob('*.npy'))[0])
animate_pose(sample_path)